# Compositionality Shared Task - ModernBERT (Simple 80/20 Split)

> **LEGACY** - uses the pre-marker 4-sequence architecture and is now superseded by `main.py` / `notebook/run_main.ipynb`.

**Modules:** shared in `src/` (dataset, model, loss, train, folds) - single source of truth for both notebooks.

In [ ]:
import os
import sys
import glob
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from transformers import AutoTokenizer, get_linear_schedule_with_warmup


def _setup_src():
    # Local runs: notebook sits next to src/ (repo root) or one level up
    for base in ('.', '..'):
        if os.path.isdir(os.path.join(base, 'src')):
            sys.path.insert(0, os.path.abspath(base))
            return
    # Kaggle runs: GitHub repo mounted under /kaggle/input/<repo>/src
    hits = sorted(glob.glob('/kaggle/input/*/src') + glob.glob('/kaggle/input/**/src', recursive=True))
    if hits:
        sys.path.insert(0, os.path.dirname(hits[0]))
        return
    raise RuntimeError('src/ not found. Add the GitHub repo as a Kaggle input (Add Input -> GitHub).')


_setup_src()

from src.constants import MODEL_NAME, MAX_LENGTH, MAX_CONTEXT_LENGTH
from src.dataset import NNDataset
from src.model import ModernBERTRegressor
from src.loss import CombinedLoss
from src.train import train_epoch, evaluate, unfreeze_top_layers

KAGGLE_PATH = '/kaggle/input/datasets/ieltsmater/compartment/Compartment'
OUTPUT_DIR = '/kaggle/working'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Load Data & 80/20 Split

Target-level GroupShuffleSplit: compounds never overlap train/val.

In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv(f'{KAGGLE_PATH}/dataset/en-nn-train.tsv', sep='\t')

# GroupSplit: Đảm bảo 100% từ ở Val KHÔNG xuất hiện ở Train
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(df, groups=df['Compound']))

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)

print(f'Train: {len(train_df)} rows | {train_df["Compound"].nunique()} unique targets')
print(f'Val:   {len(val_df)} rows | {val_df["Compound"].nunique()} unique targets')
# Kiểm tra số từ trùng lặp (Kết quả bắt buộc = 0)
overlap = set(train_df['Compound']).intersection(set(val_df['Compound']))
print(f'Số từ trùng lặp giữa Train và Val: {len(overlap)}')

## 2. Configuration

Hyperparameters differ per notebook; binaries are shared via `src/`.

In [ ]:
CONFIG = {
    'model_name': MODEL_NAME,
    'batch_size': 16,
    'head_lr': 1e-4,
    'encoder_lr': 2e-6,
    'weight_decay': 0.01,
    'num_epochs': 6,
    'freeze_epochs': 2,
    'unfreeze_from_layer': 12,  # ModernBERT-base có 22 layers -> Unfreeze layers 12-21
    'warmup_ratio': 0.1,
    'dropout': 0.1,
    'max_length': MAX_LENGTH,
    'max_context_length': MAX_LENGTH,
    # Điểm bổ sung tối ưu:
    'loss_type': 'mse_ccc',     # Cân bằng giữa MSE và Spearman/CCC
    'ccc_weight': 0.5,
    'patience': 2,              # Early stopping: chỉ đếm từ phase unfreeze
    'seed': 42
}

print('Config:', CONFIG)

## 3. Tokenizer & Loaders

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

train_dataset = NNDataset(train_df, tokenizer, max_context_length=CONFIG['max_context_length'])
val_dataset = NNDataset(val_df, tokenizer, max_context_length=CONFIG['max_context_length'])

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'] * 2,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f'Loaders ready: {len(train_loader)} train batches, {len(val_loader)} val batches')

## 4. Training (Gradual Unfreezing + MSE/CCC Loss + Early Stopping)

In [ ]:
model = ModernBERTRegressor(MODEL_NAME, dropout=CONFIG['dropout'], freeze_bert=True).to(device)
criterion = CombinedLoss(ccc_weight=CONFIG['ccc_weight'])
scaler = GradScaler('cuda')

train_losses = []
val_rhos = []

# Phase 1: Frozen Encoder (only train the two regressor heads)
head_params = list(model.mod_regressor.parameters()) + list(model.head_regressor.parameters())
optimizer = AdamW(head_params, lr=CONFIG['head_lr'], weight_decay=CONFIG['weight_decay'])

total_ph1_steps = len(train_loader) * CONFIG['freeze_epochs']
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_ph1_steps * CONFIG['warmup_ratio']),
    num_training_steps=total_ph1_steps
)

best_rho = -1
best_epoch = -1
no_improve_epochs = 0

for epoch in range(CONFIG['num_epochs']):
    # --- Phase Transition (unfreeze at epoch freeze_epochs+1) ---
    if epoch == CONFIG['freeze_epochs']:
        print(f'\n>>> Unfreezing top layers at epoch {epoch+1} <<<')
        unfreeze_top_layers(model, CONFIG['unfreeze_from_layer'])

        ph2_epochs = CONFIG['num_epochs'] - CONFIG['freeze_epochs']

        encoder_params = [p for n, p in model.named_parameters()
                          if 'mod_regressor' not in n and 'head_regressor' not in n and p.requires_grad]
        head_params = list(model.mod_regressor.parameters()) + list(model.head_regressor.parameters())

        optimizer = AdamW([
            {'params': encoder_params, 'lr': CONFIG['encoder_lr']},
            {'params': head_params, 'lr': CONFIG['head_lr']}
        ], weight_decay=CONFIG['weight_decay'])

        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(len(train_loader) * ph2_epochs * CONFIG['warmup_ratio']),
            num_training_steps=len(train_loader) * ph2_epochs
        )

    phase = 'FROZEN' if epoch < CONFIG['freeze_epochs'] else 'UNFROZEN-TOP'

    train_loss = train_epoch(model, train_loader, optimizer, scheduler, criterion, scaler, device)
    mod_pred, head_pred, mod_label, head_label = evaluate(model, val_loader, device)

    rho_mod = spearmanr(mod_label, mod_pred).statistic
    rho_head = spearmanr(head_label, head_pred).statistic
    rho_mean = (rho_mod + rho_head) / 2
    train_losses.append(train_loss)
    val_rhos.append(rho_mean)

    print(f' Epoch {epoch+1}/{CONFIG["num_epochs"]} [{phase}] | '
          f'Loss: {train_loss:.4f} | Mod ρ: {rho_mod:.4f} | Head ρ: {rho_head:.4f} | Mean ρ: {rho_mean:.4f}')

    # Save best checkpoint
    if rho_mean > best_rho:
        best_rho = rho_mean
        best_epoch = epoch + 1
        no_improve_epochs = 0
        os.makedirs(f'{OUTPUT_DIR}/models', exist_ok=True)
        torch.save(model.state_dict(), f'{OUTPUT_DIR}/models/best.pt')
        best_preds = {
            'mod': mod_pred.copy(),
            'head': head_pred.copy(),
            'mod_y': mod_label.copy(),
            'head_y': head_label.copy()
        }
    else:
        # Early stopping: only count from the unfrozen phase
        if epoch >= CONFIG['freeze_epochs']:
            no_improve_epochs += 1
            print(f'  -- No improvement {no_improve_epochs}/{CONFIG["patience"]} epochs')
            if no_improve_epochs >= CONFIG['patience']:
                print(f'\n==> EARLY STOPPING at epoch {epoch+1} (best = epoch {best_epoch}, Mean ρ = {best_rho:.4f})')
                break

print(f'\nBest: Epoch {best_epoch} | Mean ρ = {best_rho:.4f}')

## 5. Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error

# 1. Tính toán trước chỉ số Spearman & RMSE
mod_rho = spearmanr(best_preds['mod_y'], best_preds['mod']).statistic
head_rho = spearmanr(best_preds['head_y'], best_preds['head']).statistic
mean_rho = (mod_rho + head_rho) / 2

rmse_mod = np.sqrt(mean_squared_error(best_preds['mod_y'], best_preds['mod']))
rmse_head = np.sqrt(mean_squared_error(best_preds['head_y'], best_preds['head']))

# 2. Khởi tạo Biểu đồ (Sử dụng Twin Axis cho Loss & Rho)
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

# --- Subplot 1: Loss & Mean Rho Curve ---
epochs_range = range(1, len(train_losses) + 1)

# Trục Y bên trái cho Train Loss
color_loss = 'tab:blue'
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Train Loss (MSE)', color=color_loss)
l1 = axes[0].plot(epochs_range, train_losses, 'o-', color=color_loss, label='Train Loss')
axes[0].tick_params(axis='y', labelcolor=color_loss)

# Trục Y bên phải cho Val Mean Rho
ax0_twin = axes[0].twinx()
color_rho = 'tab:green'
ax0_twin.set_ylabel('Val Mean Spearman ρ', color=color_rho)
l2 = ax0_twin.plot(epochs_range, val_rhos, 's-', color=color_rho, label='Val Mean ρ')
ax0_twin.tick_params(axis='y', labelcolor=color_rho)

# Đường phân cách Epoch Unfreeze
axes[0].axvline(x=CONFIG['freeze_epochs'] + 0.5, color='r', linestyle='--', alpha=0.7, label='Unfreeze Phase')

# Gộp Legend từ cả 2 trục Y
lines = l1 + l2
labels = [l.get_label() for l in lines] + ['Unfreeze Phase']
axes[0].legend(lines + [axes[0].lines[-1]], labels, loc='center left')
axes[0].set_title('Training & Validation Convergence')
axes[0].grid(True, alpha=0.3)

# --- Subplot 2: Modifier Distribution (Gold vs Pred) ---
axes[1].hist(best_preds['mod_y'], bins=20, alpha=0.5, label='Gold', density=True, color='gray')
axes[1].hist(best_preds['mod'], bins=20, alpha=0.5, label='Predicted', density=True, color='tab:blue')
axes[1].set_title(f'Modifier: Spearman ρ = {mod_rho:.4f}')
axes[1].set_xlabel('Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# --- Subplot 3: Head Distribution (Gold vs Pred) ---
axes[2].hist(best_preds['head_y'], bins=20, alpha=0.5, label='Gold', density=True, color='gray')
axes[2].hist(best_preds['head'], bins=20, alpha=0.5, label='Predicted', density=True, color='tab:orange')
axes[2].set_title(f'Head: Spearman ρ = {head_rho:.4f}')
axes[2].set_xlabel('Score')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 3. In Báo cáo Chi tiết
print('=== BEST VALIDATION METRICS ===')
print(f'Val Mod  Spearman ρ: {mod_rho:.4f}  |  RMSE: {rmse_mod:.4f}')
print(f'Val Head Spearman ρ: {head_rho:.4f}  |  RMSE: {rmse_head:.4f}')
print(f'Mean Spearman ρ:     {mean_rho:.4f}')

## 6. Predict on Trial & Generate Submission

In [ ]:
df_trial = pd.read_csv(f'{KAGGLE_PATH}/trial/en-nn-trial.tsv', sep='\t')
trial_dataset = NNDataset(df_trial, tokenizer, max_context_length=CONFIG['max_context_length'], is_test=True)
trial_loader = DataLoader(trial_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

model.load_state_dict(torch.load(f'{OUTPUT_DIR}/models/best.pt', weights_only=True))
model.eval()

res = evaluate(model, trial_loader, device)
trial_pred_mod, trial_pred_head = res[0], res[1]

t_rho_mod = spearmanr(df_trial['ModAvg'], trial_pred_mod).statistic
t_rho_head = spearmanr(df_trial['HeadAvg'], trial_pred_head).statistic

print('=== Trial Results ===')
print(f'Mod  ρ: {t_rho_mod:.4f} | RMSE: {np.sqrt(mean_squared_error(df_trial["ModAvg"], trial_pred_mod)):.4f}')
print(f'Head ρ: {t_rho_head:.4f} | RMSE: {np.sqrt(mean_squared_error(df_trial["HeadAvg"], trial_pred_head)):.4f}')
print(f'Mean ρ: {(t_rho_mod + t_rho_head)/2:.4f}')

In [ ]:
import os
import pandas as pd

# 1. Tạo thư mục chứa file nộp bài
os.makedirs(f'{OUTPUT_DIR}/submission', exist_ok=True)

# 2. Xác định tên cột ID linh hoạt (tID hoặc ContextID)
id_col = 'tID' if 'tID' in df_trial.columns else ('ContextID' if 'ContextID' in df_trial.columns else df_trial.columns[0])

# 3. Đóng gói DataFrame theo đúng định dạng kết quả dự đoán
submission = pd.DataFrame({
    'tID': df_trial[id_col],
    'Modifier': trial_pred_mod,
    'Head': trial_pred_head
})

# 4. Xuất file TSV chuẩn (No Index, No Header theo quy ước SemEval)
sub_path = f'{OUTPUT_DIR}/submission/en-nn-trial-pred.tsv'
submission.to_csv(sub_path, sep='\t', index=False, header=False)

print(f'✅ Đã lưu file submission thành công tại: {sub_path}')
print('\n=== Xem trước 5 dòng đầu tiên ===')
print(submission.head().to_string(index=False))
print(f'\nTổng số bản ghi: {len(submission)} rows')